In [41]:
import pandas as pd

customers = pd.read_csv("../data/raw/customers.csv")
order_items = pd.read_csv("../data/raw/order_items.csv")
orders = pd.read_csv("../data/raw/orders.csv")
products = pd.read_csv("../data/raw/products.csv")

print(customers.head())
print(orders.head())
print(order_items.head())
print(products.head())

   customer_id name gender  age city signup_date
0            1  김수민      F   19   광주  2024-08-14
1            2  김정호      F   32   대구  2025-12-28
2            3  이경수      F   61   성남  2024-08-07
3            4  조영호      F   55   울산  2026-06-08
4            5  이예원      F   19   부산  2024-11-08
   order_id  customer_id  order_date payment_method order_status
0         1          123  2026-07-02           card    completed
1         2           77  2025-09-17      naver_pay    cancelled
2         3          138  2026-01-14  bank_transfer    cancelled
3         4           57  2026-03-27      kakao_pay    cancelled
4         5          125  2026-02-15           card    cancelled
   order_item_id  order_id  product_id  quantity  unit_price
0              1         1         100         3      102000
1              2         1          87         5       25000
2              3         1           7         3      142000
3              4         1           9         3      193000
4          

In [42]:
# 원본 구조에 대한 이력(Evidence)을 만든다.

raw_data = {
    "customers":customers,
    "order_items":order_items,
    "products":products,
    "orders":orders
}

print(raw_data)

{'customers':      customer_id name gender  age city signup_date
0              1  김수민      F   19   광주  2024-08-14
1              2  김정호      F   32   대구  2025-12-28
2              3  이경수      F   61   성남  2024-08-07
3              4  조영호      F   55   울산  2026-06-08
4              5  이예원      F   19   부산  2024-11-08
..           ...  ...    ...  ...  ...         ...
145          146  김숙자      M   61   성남  2026-02-17
146          147  이정남      M   19   부산  2025-04-08
147          148  오도현      M   29   고양  2026-08-10
148          149  김정자      M   20   부산  2024-12-14
149          150  조미영      M   40   대전  2026-01-29

[150 rows x 6 columns], 'order_items':      order_item_id  order_id  product_id  quantity  unit_price
0                1         1         100         3      102000
1                2         1          87         5       25000
2                3         1           7         3      142000
3                4         1           9         3      193000
4                5 

In [43]:
summary_list = []
for name, frame in raw_data.items():
    info = {
        "dataset" : name,
        "rows" : frame.shape[0],
        "columns" : frame.shape[1],
        "missing_values" : (frame.isna().sum()).sum(),
        "duplicated_rows" : frame.duplicated().sum(),
    }
    summary_list.append(info)

summary_list


[{'dataset': 'customers',
  'rows': 150,
  'columns': 6,
  'missing_values': np.int64(0),
  'duplicated_rows': np.int64(0)},
 {'dataset': 'order_items',
  'rows': 764,
  'columns': 5,
  'missing_values': np.int64(0),
  'duplicated_rows': np.int64(0)},
 {'dataset': 'products',
  'rows': 100,
  'columns': 4,
  'missing_values': np.int64(0),
  'duplicated_rows': np.int64(0)},
 {'dataset': 'orders',
  'rows': 300,
  'columns': 5,
  'missing_values': np.int64(0),
  'duplicated_rows': np.int64(0)}]

In [44]:
from src.preprocessing import (
    compare_shapes,
    preprocess_sales_data,
    validate_relationships,
)

processed_data = preprocess_sales_data(raw_data)

preprocessing_comparison = compare_shapes(
    raw_data,
    processed_data,
)

# preprocessing_comparison

print(raw_data["orders"].head())
print(processed_data["orders"].head())

relationship_checks = validate_relationships(
    processed_data
)

   order_id  customer_id  order_date payment_method order_status
0         1          123  2026-07-02           card    completed
1         2           77  2025-09-17      naver_pay    cancelled
2         3          138  2026-01-14  bank_transfer    cancelled
3         4           57  2026-03-27      kakao_pay    cancelled
4         5          125  2026-02-15           card    cancelled
   order_id  customer_id order_date payment_method order_status order_month  \
0         1          123 2026-07-02           card    completed     2026-07   
1         2           77 2025-09-17      naver_pay    cancelled     2025-09   
2         3          138 2026-01-14  bank_transfer    cancelled     2026-01   
3         4           57 2026-03-27      kakao_pay    cancelled     2026-03   
4         5          125 2026-02-15           card    cancelled     2026-02   

  order_dayofweek  
0        Thursday  
1       Wednesday  
2       Wednesday  
3          Friday  
4          Sunday  


In [45]:
processed_data["orders"]["order_dayofweek"].value_counts()

order_dayofweek
Saturday     50
Tuesday      47
Monday       47
Wednesday    44
Sunday       43
Friday       35
Thursday     34
Name: count, dtype: int64

In [46]:
key_map = {
    "customers" : "customer_id",
    "products" : "product_id",
    "orders" : "order_id",
    "order_items" : "order_item_id",
}

In [47]:
pk_check = []

for dataset, key in key_map.items():
    pk_check.append(
        {
            "dataset" : dataset,
            "key" : key,
            "missing_count" : processed_data[dataset][key].isna().sum(),
            "duplicated_count" : processed_data[dataset][key].duplicated().sum(),
            "status" : (
                "PASS"
                if processed_data[dataset][key].isna().sum() == 0 and processed_data[dataset][key].duplicated().sum() == 0
                else
                    "FAIL"
                ),
        }
    )

pk_check

[{'dataset': 'customers',
  'key': 'customer_id',
  'missing_count': np.int64(0),
  'duplicated_count': np.int64(0),
  'status': 'PASS'},
 {'dataset': 'products',
  'key': 'product_id',
  'missing_count': np.int64(0),
  'duplicated_count': np.int64(0),
  'status': 'PASS'},
 {'dataset': 'orders',
  'key': 'order_id',
  'missing_count': np.int64(0),
  'duplicated_count': np.int64(0),
  'status': 'PASS'},
 {'dataset': 'order_items',
  'key': 'order_item_id',
  'missing_count': np.int64(0),
  'duplicated_count': np.int64(0),
  'status': 'PASS'}]

In [72]:
order_sales = order_items.merge(
    orders[
        [
            "order_id",
            "customer_id",
            "order_date",
            "order_status",
        ]
    ],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

order_sales.head()


,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,order_status,_merge
0,1,1,100,3,102000,306000,123,2026-07-02,completed,both
1,2,1,87,5,25000,125000,123,2026-07-02,completed,both
2,3,1,7,3,142000,426000,123,2026-07-02,completed,both
3,4,1,9,3,193000,579000,123,2026-07-02,completed,both
4,5,2,72,4,189000,756000,77,2025-09-17,cancelled,both


In [49]:
print("병합 전 행 수 : ", len(order_items))
print("병합 후 행 수 : ", len(order_sales))

병합 전 행 수 :  764
병합 후 행 수 :  764


In [50]:
if "line_total" not in order_items.columns:
    order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 764 entries, 0 to 763
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   order_item_id  764 non-null    int64
 1   order_id       764 non-null    int64
 2   product_id     764 non-null    int64
 3   quantity       764 non-null    int64
 4   unit_price     764 non-null    int64
 5   line_total     764 non-null    int64
dtypes: int64(6)
memory usage: 35.9 KB


In [76]:
completed_order_sales = order_sales.loc[
    order_sales["order_status"].eq("completed")
]

print(completed_order_sales.head())
print(completed_order_sales["order_status"].value_counts())

    order_item_id  order_id  product_id  quantity  unit_price  line_total  \
0               1         1         100         3      102000      306000   
1               2         1          87         5       25000      125000   
2               3         1           7         3      142000      426000   
3               4         1           9         3      193000      579000   
12             13         6          83         3       24000       72000   

    customer_id  order_date order_status _merge  
0           123  2026-07-02    completed   both  
1           123  2026-07-02    completed   both  
2           123  2026-07-02    completed   both  
3           123  2026-07-02    completed   both  
12           87  2026-05-16    completed   both  
order_status
completed    474
Name: count, dtype: int64


In [60]:
print("전체 주문 건수 :", len(order_sales))
print("전체 주문 건수(completed) :", len(order_sales[order_sales["order_status"]=="completed"]))
print("전체 주문 건수(completed 외) :", len(order_sales[order_sales["order_status"]!="completed"]))

전체 주문 건수 : 764
전체 주문 건수(completed) : 474
전체 주문 건수(completed 외) : 290


In [64]:
products

,product_id,product_name,category,price
0,1,전자기기 상품 001,전자기기,160000
1,2,도서 상품 002,도서,34000
2,3,전자기기 상품 003,전자기기,152000
3,4,생활용품 상품 004,생활용품,70000
4,5,식품 상품 005,식품,186000
...,...,...,...,...
95,96,생활용품 상품 096,생활용품,112000
96,97,전자기기 상품 097,전자기기,154000
97,98,스포츠 상품 098,스포츠,10000
98,99,뷰티 상품 099,뷰티,200000


In [81]:
completed_order_sales = completed_order_sales.merge(
    products[
        ["product_id", "category"]
    ],
    on="product_id",
    how="left",
    validate="many_to_one"
)

In [87]:
print(completed_order_sales.head())

   order_item_id  order_id  product_id  quantity  unit_price  line_total  \
0              1         1         100         3      102000      306000   
1              2         1          87         5       25000      125000   
2              3         1           7         3      142000      426000   
3              4         1           9         3      193000      579000   
4             13         6          83         3       24000       72000   

   customer_id  order_date order_status _merge category  
0          123  2026-07-02    completed   both       도서  
1          123  2026-07-02    completed   both       도서  
2          123  2026-07-02    completed   both       도서  
3          123  2026-07-02    completed   both      스포츠  
4           87  2026-05-16    completed   both     전자기기  


In [88]:
category_sales = (
    completed_order_sales
    .groupby("category", as_index=False)
    .agg(
        total_quantity=("quantity", "sum"),
        total_sales=("line_total", "sum"),
    )
)

print(category_sales)

  category  total_quantity  total_sales
0       도서             149     16389000
1       뷰티             223     23383000
2     생활용품             272     23915000
3      스포츠             295     31743000
4       식품             133     16573000
5     전자기기             259     26400000
6       패션             111     10587000
